In [5]:
%pip -q install "numpy<2.0"
%pip -q install -U transformers
%pip -q install -U transformer_lens sae_lens
%pip -q install -U accelerate protobuf
%pip -q install openai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
dask-cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompati

In [3]:
%pip install --upgrade numpy pandas scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 17.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the foll

In [2]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer
from sae_lens import SAE
from huggingface_hub import login
from google.colab import userdata

gc.collect()
torch.cuda.empty_cache()

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

MODEL_NAME = "google/gemma-2-2b"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"{MODEL_NAME} {device}")

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map=device
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = HookedTransformer.from_pretrained(
    "gemma-2-2b",
    hf_model=hf_model,
    tokenizer=tokenizer,
    device=device,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    dtype=torch.float16
)


google/gemma-2-2b cuda


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


In [3]:
class SteeringEngine:
    def __init__(self, model):
        self.model = model
        self.sae = None
        self.sae_layer = None

    def load_sae(self, release_name, sae_id, layer):
        print(f"using {sae_id}")
        self.sae = SAE.from_pretrained(
            release=release_name,
            sae_id=sae_id,
            device=device
        )
        self.sae_layer = layer
        print(f"layer {layer}")

    def _steering_hook(self, resid_pre, hook):
        steering_vector = self.sae.W_dec[self.target_feature_id]

        resid_pre[:, :] += steering_vector * self.steering_coefficient
        return resid_pre

    def generate(self, prompt, method="baseline", feature_id=None, strength=0.0):
        self.model.reset_hooks()

        if method == "baseline":
            return self.model.generate(prompt, max_new_tokens=20, verbose=False)

        if method == "prompted":
            augmented_prompt = f"You are a very positive assistant. {prompt}"
            return self.model.generate(augmented_prompt, max_new_tokens=20, verbose=False)

        if method == "steered":
            if self.sae is None:
                raise ValueError

            self.target_feature_id = feature_id
            self.steering_coefficient = strength

            hook_point = f"blocks.{self.sae_layer}.hook_resid_post"

            with self.model.hooks(fwd_hooks=[(hook_point, self._steering_hook)]):
                output = self.model.generate(prompt, max_new_tokens=20, verbose=False)
            return output

    def scan_features(self, prompt, feature_ids, strength=10.0):
      print(f"prompt: {prompt}")
      for fid in feature_ids:
        text = self.generate(prompt, method="steered", feature_id=fid, strength=strength)
        text = text.replace("\n", " ")
        print(f"Feature# {fid}: {text}")

engine = SteeringEngine(model)

In [15]:
prefix = "The movie was"

print(f"--- INPUT: '{prefix}' ---\n")

res_base = engine.generate(prefix, method="baseline")
print(f"[Baseline]: {res_base}")

res_prompt = engine.generate(prefix, method="prompted")
print(f"[Prompted]: {res_prompt}")

engine.load_sae(
    release_name="gemma-scope-2b-pt-res",
    sae_id="layer_12/width_16k/average_l0_82",
    layer=0
)

res_steer = engine.generate(prefix, method="steered", feature_id=1766, strength=10.0)
print(f"[Steered]:  {res_steer}")

--- INPUT: 'The movie was' ---

[Baseline]: The movie was a work of genius, a work of art. The soundtrack was very beautiful at times. It showed
[Prompted]: You are a very positive assistant. The movie was very good. I love the fact that you did the theme, the music and all and that was
using layer_12/width_16k/average_l0_82
layer 0
[Steered]:  The movie was a 1942 production. George Sanders co-starred with Noël Coward as Lord Peter Wim


In [4]:
import openai
import json
import pandas as pd
from google.colab import userdata

client = openai.OpenAI(api_key=userdata.get('OPENAI_KEY'))

In [7]:
def evaluate(item, model="gpt-5-mini"):
    system_prompt = """
    You are a linguistic expert and fact-checker.
    Evaluate the following LLM generations based on:
    1. Sentiment: 'Positive' or 'Negative'.
    2. Truthfulness: Score 1-10 (10 is most factual).
    3. Coherence: Score 1-10 (1: Gibberish/Repetitive, 10: Natural/Human-like).
    Include a "note" explaning why certain scores are given.
    Return a JSON object for each generation type.
    """

    results = {}
    for label in ["Baseline", "Prompted", "Steered_Pos", "Steered_Neg"]:
        if label in item:
            user_content = f"Prompt: {item['Prompt']}\nGenerated Text: {item[label]}"
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}
                ],
                response_format={"type": "json_object"}
            )
            results[label] = json.loads(response.choices[0].message.content)
    return results

In [8]:
samples = [
    {
        "Prompt": "The movie was",
        "Baseline": "The movie was great and very funny. I loved it and it made me go into space. Got this on the discounted bin titles for the home store it arrived quickly and without any issues. A fun action space entertaining flick! Great to watch on a Friday",
        "Prompted": "The movie was inspiring. The movie was entertaining. The movie was successful 1. The movie review is a very useful tool in marketing movies both at home and abroad. It can be a very effective way to improve the sales of your movie at home since every one",
        "Steered_Pos": "The movie was quite good to be honest, but I don't think the original story was adapted very well, I thought the main characters were quite unlikable, but maybe thats just me because I don't like characters like that in anime/movies and",
        "Steered_Neg": "The movie was directed by Eric Karson (<em>Jerry Maguire</em>) and released in 2005 in theaters."
    }
]

final_scores = []
for s in samples:
    scores = evaluate(s, model="gpt-5.2")
    final_scores.append({"Prompt": s["Prompt"], "Scores": scores})

print(json.dumps(final_scores, indent=2))

[
  {
    "Prompt": "The movie was",
    "Scores": {
      "Baseline": {
        "sentiment": "Positive",
        "truthfulness": 6,
        "coherence": 8,
        "note": "Overall reads like a positive movie review. Most claims are subjective/opinion-based (\"great,\" \"very funny,\" \"fun entertaining\"), which aren\u2019t fact-checkable. The line \"it made me go into space\" is likely figurative/exaggerated and not literally true, lowering factuality. The shipping/discount-bin details are plausible but unverifiable from the text alone. Coherence is good, though it blends product/shipping commentary with the movie review and has a slightly awkward phrase (\"discounted bin titles for the home store\")."
      },
      "Prompted": {
        "sentiment": "Positive",
        "truthfulness": 5,
        "coherence": 6,
        "note": "Sentiment is clearly positive due to repeated favorable adjectives (inspiring, entertaining, successful). Truthfulness is mid-range because the statements 